<a href="https://colab.research.google.com/github/aryananand0/Math-344-Lab-Project-Fall-2026/blob/main/Notebooks/04_what_it_learned_my_ratings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 4: What Did It Learn, and What Would It Recommend to You?
**MATH 344L — Applied Linear Algebra Lab | Fall 2026**

In Milestone 3, Model 2 did better than the baseline. This time I am looking at what the SVD learned from movie ratings and how it changes recommendations based on ten movies I rated.

## 1. Setup

In [1]:
from pathlib import Path
import subprocess
import sys

# Use the local files if I opened the notebook from my checkout
repo = Path.cwd()
if not (repo / "Src").is_dir():
    repo = repo.parent
if not (repo / "Src").is_dir():
    repo = Path("Math-344-Lab-Project-Fall-2026")
    if not (repo / "Src").is_dir():
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/aryananand0/Math-344-Lab-Project-Fall-2026.git", str(repo)],
            check=True
        )

sys.path.insert(0, str(repo.resolve() / "Src"))

import numpy as np
import pandas as pd

from evaluate import to_matrix
from models import baseline_matrix, rank_k

I use the same matrix, baseline, and rank-k functions from the earlier notebooks.

In [2]:
import urllib.request
import zipfile

# Download MovieLens if either file is missing
if not (Path("ml-100k/u.data").is_file() and Path("ml-100k/u.item").is_file()):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip"
    )
    with zipfile.ZipFile("ml-100k.zip") as zipped:
        zipped.extractall()

ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "item", "rating", "time"]
)
titles = pd.read_csv(
    "ml-100k/u.item", sep="|", encoding="latin-1",
    header=None, usecols=[1]
)[1].to_numpy()

print(len(titles), titles[49])
print("Ratings:", len(ratings))

1682 Star Wars (1977)
Ratings: 100000


The ratings file tells us which movies people rated and how many stars they gave them. I also loaded movie titles so the output shows names instead of just movie IDs. Movie ID 50 is at index 49 because Python starts at 0.

## 2. Fit Model 2 using all the ratings

In [3]:
# We can use all the ratings now because we are exploring the model
R = to_matrix(ratings)
known = ~np.isnan(R)
B = baseline_matrix(R)

# Subtract the baseline and keep missing residuals at 0
X = np.where(known, R - B, 0.0)
U_all, s_all, Vt_all = np.linalg.svd(X, full_matrices=False)

# Only study movies with at least 50 actual ratings
n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("Popular movies:", popular.sum())

Popular movies: 603


For Milestone 3 I used a training set and a hidden test set to check prediction accuracy. Here I use all the ratings because I am trying to understand the final model. I only look at movies with 50 or more ratings so a movie with just a few ratings does not throw off the results.

## 3. Look at the hidden factors

In [4]:
def show_factor(f, n=6):
    # Each row in Vt gives the movie scores for one factor
    v = Vt_all[f - 1]

    # Sort only the movies with at least 50 ratings
    idx = np.where(popular)[0]
    order = idx[np.argsort(v[idx])]

    print(f"Factor {f}, one end:")
    for m in order[:n]:
        print(f"  {v[m]:+.3f} {titles[m]}")

    print(f"Factor {f}, other end:")
    for m in order[::-1][:n]:
        print(f"  {v[m]:+.3f} {titles[m]}")

# Look at the first and third factors from the assignment
show_factor(1)
print()
show_factor(3)

Factor 1, one end:
  -0.087 Clockwork Orange, A (1971)
  -0.079 Trainspotting (1996)
  -0.072 Leaving Las Vegas (1995)
  -0.072 Pulp Fiction (1994)
  -0.069 Brazil (1985)
  -0.067 Fargo (1996)
Factor 1, other end:
  +0.211 Independence Day (ID4) (1996)
  +0.175 Return of the Jedi (1983)
  +0.149 Raiders of the Lost Ark (1981)
  +0.149 Back to the Future (1985)
  +0.136 Braveheart (1995)
  +0.134 Forrest Gump (1994)

Factor 3, one end:
  -0.137 Independence Day (ID4) (1996)
  -0.131 Star Trek: First Contact (1996)
  -0.130 Rock, The (1996)
  -0.129 Alien (1979)
  -0.121 Monty Python and the Holy Grail (1974)
  -0.119 Aliens (1986)
Factor 3, other end:
  +0.163 Sound of Music, The (1965)
  +0.144 Grease (1978)
  +0.099 E.T. the Extra-Terrestrial (1982)
  +0.087 Mary Poppins (1964)
  +0.085 Jaws 2 (1978)
  +0.084 Snow White and the Seven Dwarfs (1937)


Each hidden factor puts movies at two ends of a scale. Looking at the movies with the lowest and highest scores can help me figure out what that factor might represent. The plus and minus signs could swap without changing the SVD, so I pay more attention to the movie groups than the signs.

## 4. What would it recommend to me?

In [5]:
def find(word):
    # Search movie names so I can find the right IDs for my ratings
    for m, title in enumerate(titles):
        if word.lower() in title.lower():
            print(m + 1, title)

find("alien")

176 Aliens (1986)
183 Alien (1979)
343 Alien: Resurrection (1997)
665 Alien 3 (1992)


This searches movie names without worrying about capital letters. It prints the movie ID I need for the ratings dictionary below.

In [6]:
my_ratings = {
    50: 5,   # Star Wars
    172: 5,  # The Empire Strikes Back
    183: 4,  # Alien
    195: 5,  # The Terminator
    89: 4,   # Blade Runner
    1: 5,    # Toy Story
    143: 3,  # The Sound of Music
    451: 3,  # Grease
    88: 3,   # Sleepless in Seattle
    739: 3   # Pretty Woman
}


# Print the movies so I can double-check the IDs and ratings
assert 10 <= len(my_ratings) <= 15, "Rate between 10 and 15 movies."
assert all(1 <= movie <= len(titles) for movie in my_ratings)
assert all(isinstance(stars, int) and 1 <= stars <= 5
           for stars in my_ratings.values())
for movie, stars in my_ratings.items():
    print(f"{movie:4d}  {stars} stars  {titles[movie - 1]}")

  50  5 stars  Star Wars (1977)
 172  5 stars  Empire Strikes Back, The (1980)
 183  4 stars  Alien (1979)
 195  5 stars  Terminator, The (1984)
  89  4 stars  Blade Runner (1982)
   1  5 stars  Toy Story (1995)
 143  3 stars  Sound of Music, The (1965)
 451  3 stars  Grease (1978)
  88  3 stars  Sleepless in Seattle (1993)
 739  3 stars  Pretty Woman (1990)


The dictionary stores my ratings for ten movies I have seen. The movie IDs are the keys and the values are my ratings from 1 to 5 stars.

In [7]:
# Make one new row for my ratings, leaving other movies as missing
me = np.full((1, R.shape[1]), np.nan)
for movie, stars in my_ratings.items():
    me[0, movie - 1] = stars

# Add the new user under the 943 users already in the dataset
R_me = np.vstack([R, me])
known_me = ~np.isnan(R_me)
B_me = baseline_matrix(R_me)
X_me = np.where(known_me, R_me - B_me, 0.0)

# Refit the SVD and use Model 2 with k = 10 (as in the handout)
U_me, s_me, Vt_me = np.linalg.svd(X_me, full_matrices=False)
P = rank_k(U_me, s_me, Vt_me, 10) + B_me
print("Matrix shape:", R_me.shape)

Matrix shape: (944, 1682)


I added my ratings as a new row, making me user 944 in the matrix. Then I ran Model 2 again with my ratings included. I used k = 10 to follow the Milestone 4 handout, and I compare other values of k in Exercise 4.

In [8]:
# The last row contains the predictions for the new user
mine = P[-1]
base = B_me[-1]

# Recommend only popular movies that the new user has not rated
can_pick = popular & np.isnan(R_me[-1])

def top(scores, n=10):
    # Put movies we cannot recommend at the very bottom
    possible = np.where(can_pick, scores, -np.inf)
    return np.argsort(-possible)[:n]

model_top = top(mine)
baseline_top = top(base)
overlap = len(set(model_top) & set(baseline_top))

print("Model 2 recommends:")
for m in model_top:
    print(f"  {mine[m]:.2f} {titles[m]}")

print("\nBaseline recommends:")
for m in baseline_top:
    print(f"  {base[m]:.2f} {titles[m]}")

print("\nIn both top 10 lists:", overlap)

Model 2 recommends:
  4.95 Close Shave, A (1995)
  4.88 Wrong Trousers, The (1993)
  4.81 Casablanca (1942)
  4.81 Schindler's List (1993)
  4.79 Shawshank Redemption, The (1994)
  4.79 Wallace & Gromit: The Best of Aardman Animation (1996)
  4.75 Usual Suspects, The (1995)
  4.74 Good Will Hunting (1997)
  4.71 Titanic (1997)
  4.71 Godfather, The (1972)

Baseline recommends:
  4.93 Close Shave, A (1995)
  4.85 Wrong Trousers, The (1993)
  4.83 Schindler's List (1993)
  4.83 Shawshank Redemption, The (1994)
  4.79 Casablanca (1942)
  4.78 Wallace & Gromit: The Best of Aardman Animation (1996)
  4.78 Usual Suspects, The (1995)
  4.76 Good Will Hunting (1997)
  4.71 Titanic (1997)
  4.71 12 Angry Men (1957)

In both top 10 lists: 9


I compared Model 2 with the baseline to see whether SVD changes my recommendations. Movies I already rated and movies with fewer than 50 ratings are left out. The two lists have 9 movies in common, so most of the recommendations are still the same popular choices.

In [9]:
# See the part of my prediction that came from SVD
personal = mine - base

print("Moved up most for me:")
for m in top(personal, 6):
    print(f"  {personal[m]:+.2f} {titles[m]}")

print("\nMoved down most for me:")
for m in top(-personal, 6):
    print(f"  {personal[m]:+.2f} {titles[m]}")

Moved up most for me:
  +0.12 Fargo (1996)
  +0.09 Return of the Jedi (1983)
  +0.08 Scream (1996)
  +0.07 Beavis and Butt-head Do America (1996)
  +0.06 Independence Day (ID4) (1996)
  +0.06 2001: A Space Odyssey (1968)

Moved down most for me:
  -0.11 English Patient, The (1996)
  -0.09 Braveheart (1995)
  -0.07 Four Weddings and a Funeral (1994)
  -0.07 Ghost (1990)
  -0.07 Dead Poets Society (1989)
  -0.07 Dirty Dancing (1987)


Subtracting the baseline from Model 2 shows what SVD adds for me. Positive values move a movie up, and negative values move it down. The changes here are fairly small, but they show how my ratings affect the recommendations.

## 5. One main finding

In [10]:
# Make a summary using the results from this run
up = top(personal, 1)[0]
down = top(-personal, 1)[0]
print(
    f"With {len(my_ratings)} ratings, Model 2 and the baseline agreed on "
    f"{overlap} of their top 10 recommendations. "
    f"The SVD moved {titles[up]} up by {personal[up]:.2f} stars "
    f"and {titles[down]} down by {abs(personal[down]):.2f} stars."
)

With 10 ratings, Model 2 and the baseline agreed on 9 of their top 10 recommendations. The SVD moved Fargo (1996) up by 0.12 stars and English Patient, The (1996) down by 0.11 stars.


---
## Exercises

### Exercise 1. Name factors 1, 2, and 3

In [11]:
# Show all three factors together so I can compare them
for f in [1, 2, 3]:
    show_factor(f)
    print()

Factor 1, one end:
  -0.087 Clockwork Orange, A (1971)
  -0.079 Trainspotting (1996)
  -0.072 Leaving Las Vegas (1995)
  -0.072 Pulp Fiction (1994)
  -0.069 Brazil (1985)
  -0.067 Fargo (1996)
Factor 1, other end:
  +0.211 Independence Day (ID4) (1996)
  +0.175 Return of the Jedi (1983)
  +0.149 Raiders of the Lost Ark (1981)
  +0.149 Back to the Future (1985)
  +0.136 Braveheart (1995)
  +0.134 Forrest Gump (1994)

Factor 2, one end:
  -0.099 Independence Day (ID4) (1996)
  -0.092 Twister (1996)
  -0.077 Eraser (1996)
  -0.074 Air Force One (1997)
  -0.067 Saint, The (1997)
  -0.066 Phenomenon (1996)
Factor 2, other end:
  +0.203 Star Wars (1977)
  +0.199 Pulp Fiction (1994)
  +0.173 Godfather, The (1972)
  +0.168 Empire Strikes Back, The (1980)
  +0.151 Fargo (1996)
  +0.146 Return of the Jedi (1983)

Factor 3, one end:
  -0.137 Independence Day (ID4) (1996)
  -0.131 Star Trek: First Contact (1996)
  -0.130 Rock, The (1996)
  -0.129 Alien (1979)
  -0.121 Monty Python and the Holy Gra

**Answer:** I would call **Factor 1 “darker movies vs. big crowd-pleasers.”** Movies like *A Clockwork Orange*, *Trainspotting*, and *Pulp Fiction* are at one end, while *Independence Day*, *Return of the Jedi*, and *Back to the Future* are at the other.

**Factor 2 is “1990s action hits vs. popular classics and fan favorites.”** One end has *Twister*, *Eraser*, and *Air Force One*, while the other has *Star Wars*, *The Godfather*, and *Pulp Fiction*. **Factor 3 is “sci-fi/action vs. musicals and family movies.”** *Alien* and *Star Trek: First Contact* are on one end, while *The Sound of Music*, *Grease*, and *Mary Poppins* are on the other.

Factor 2 was the hardest to name because its movies do not all fit into one genre. That makes sense because SVD only looks at patterns in ratings. It does not know genres, so a factor can pick up several different things at once.

### Exercise 2. My recommendations

In [12]:
# Show the overlap and a few recommendations for my written answer
print(f"Top 10 overlap: {overlap}/10")
print("\nModel 2 top 10:")
for m in model_top:
    print(f"  {mine[m]:.2f} {titles[m]}")
print("\nBiggest moves up:")
for m in top(personal, 6):
    print(f"  {personal[m]:+.2f} {titles[m]}")
print("\nBiggest moves down:")
for m in top(-personal, 6):
    print(f"  {personal[m]:+.2f} {titles[m]}")

Top 10 overlap: 9/10

Model 2 top 10:
  4.95 Close Shave, A (1995)
  4.88 Wrong Trousers, The (1993)
  4.81 Casablanca (1942)
  4.81 Schindler's List (1993)
  4.79 Shawshank Redemption, The (1994)
  4.79 Wallace & Gromit: The Best of Aardman Animation (1996)
  4.75 Usual Suspects, The (1995)
  4.74 Good Will Hunting (1997)
  4.71 Titanic (1997)
  4.71 Godfather, The (1972)

Biggest moves up:
  +0.12 Fargo (1996)
  +0.09 Return of the Jedi (1983)
  +0.08 Scream (1996)
  +0.07 Beavis and Butt-head Do America (1996)
  +0.06 Independence Day (ID4) (1996)
  +0.06 2001: A Space Odyssey (1968)

Biggest moves down:
  -0.11 English Patient, The (1996)
  -0.09 Braveheart (1995)
  -0.07 Four Weddings and a Funeral (1994)
  -0.07 Ghost (1990)
  -0.07 Dead Poets Society (1989)
  -0.07 Dirty Dancing (1987)


**Answer:** Model 2 and the baseline have **9 out of 10 movies in common** for me. The changes mostly make sense based on my ratings: *Fargo* goes up 0.12 stars and *Return of the Jedi* goes up 0.09, while *The English Patient* goes down 0.11 and *Dirty Dancing* also goes down.

One recommendation I would question is **Titanic**, which appears in the Model 2 top 10 even though I gave higher ratings to sci-fi and action movies. I think it appears because it has strong overall ratings, and the model only has ten of my ratings to work with.

### Exercise 3. A friend with opposite taste

In [13]:
# Change 5-star ratings to 1 and 1-star ratings to 5
# Keep 2, 3, and 4 the same, as the exercise says
opposite_ratings = {
    movie: (1 if stars == 5 else 5 if stars == 1 else stars)
    for movie, stars in my_ratings.items()
}

# Add the opposite user to the original data and refit Model 2
friend = np.full((1, R.shape[1]), np.nan)
for movie, stars in opposite_ratings.items():
    friend[0, movie - 1] = stars

R_friend = np.vstack([R, friend])
known_friend = ~np.isnan(R_friend)
B_friend = baseline_matrix(R_friend)
X_friend = np.where(known_friend, R_friend - B_friend, 0.0)
U_friend, s_friend, Vt_friend = np.linalg.svd(X_friend, full_matrices=False)
P_friend = rank_k(U_friend, s_friend, Vt_friend, 10) + B_friend

friend_mine = P_friend[-1]
friend_base = B_friend[-1]
friend_personal = friend_mine - friend_base
friend_can_pick = popular & np.isnan(R_friend[-1])

def friend_top(scores, n=10):
    # Same rule: popular movies that the friend has not rated
    return np.argsort(-np.where(friend_can_pick, scores, -np.inf))[:n]

print("Friend moved up most:")
for m in friend_top(friend_personal, 6):
    print(f"  {friend_personal[m]:+.2f} {titles[m]}")
print("\nFriend moved down most:")
for m in friend_top(-friend_personal, 6):
    print(f"  {friend_personal[m]:+.2f} {titles[m]}")

Friend moved up most:
  +0.23 English Patient, The (1996)
  +0.18 Pulp Fiction (1994)
  +0.13 First Wives Club, The (1996)
  +0.12 Dirty Dancing (1987)
  +0.12 Phenomenon (1996)
  +0.12 Dead Man Walking (1995)

Friend moved down most:
  -0.68 Return of the Jedi (1983)
  -0.42 Raiders of the Lost Ark (1981)
  -0.23 Liar Liar (1997)
  -0.19 Princess Bride, The (1987)
  -0.18 Jaws (1975)
  -0.18 Star Trek: The Wrath of Khan (1982)


I created a friend with the opposite ratings by swapping my 5-star and 1-star ratings. Ratings of 2, 3, or 4 stay the same, which follows the instructions for this exercise. Then I ran the model again to see whether the personal adjustments changed.

In [14]:
# See which movies switched direction between the two users
my_up = set(top(personal, 6))
my_down = set(top(-personal, 6))
friend_up = set(friend_top(friend_personal, 6))
friend_down = set(friend_top(-friend_personal, 6))

swapped_to_down = my_up & friend_down
swapped_to_up = my_down & friend_up

print("Moved up for me but down for my friend:")
for m in sorted(swapped_to_down):
    print(" ", titles[m])
if not swapped_to_down:
    print("  None of the top six crossed lists")

print("\nMoved down for me but up for my friend:")
for m in sorted(swapped_to_up):
    print(" ", titles[m])
if not swapped_to_up:
    print("  None of the top six crossed lists")

my_baseline_top = set(top(base))
friend_baseline_top = set(friend_top(friend_base))
print("\nBaseline top 10 overlap:", len(my_baseline_top & friend_baseline_top))
print("Baseline lists exactly the same:", my_baseline_top == friend_baseline_top)

Moved up for me but down for my friend:
  Return of the Jedi (1983)

Moved down for me but up for my friend:
  Dirty Dancing (1987)
  English Patient, The (1996)

Baseline top 10 overlap: 10
Baseline lists exactly the same: True


**Answer:** *Return of the Jedi* moved up for me but down for the opposite user. *Dirty Dancing* and *The English Patient* moved down for me but up for the opposite user.

The baseline top 10 **did not change at all: all 10 movies were the same**. The baseline mainly uses overall movie ratings and a user rating effect, which shifts scores together instead of changing the movie order. SVD can change the order because it picks up differences in what each user likes.

### Exercise 4. Does k matter here?

In [15]:
# We already computed the SVD for my new user, so we can reuse it
# Only k changes here; the movie ratings stay the same
moves_by_k = {}
for k in [2, 10, 50]:
    P_k = rank_k(U_me, s_me, Vt_me, k) + B_me
    moves_by_k[k] = P_k[-1] - base
    print(f"k = {k}, moved up most:")
    for m in top(moves_by_k[k], 6):
        print(f"  {moves_by_k[k][m]:+.2f} {titles[m]}")
    print()

# Check how many moved-up movies each version has in common
print("Shared moved-up movies, k=2 and k=10:",
      len(set(top(moves_by_k[2], 6)) & set(top(moves_by_k[10], 6))))
print("Shared moved-up movies, k=2 and k=50:",
      len(set(top(moves_by_k[2], 6)) & set(top(moves_by_k[50], 6))))

k = 2, moved up most:
  +0.04 Pulp Fiction (1994)
  +0.03 Fargo (1996)
  +0.03 Godfather, The (1972)
  +0.03 Leaving Las Vegas (1995)
  +0.02 2001: A Space Odyssey (1968)
  +0.02 Raising Arizona (1987)

k = 10, moved up most:
  +0.12 Fargo (1996)
  +0.09 Return of the Jedi (1983)
  +0.08 Scream (1996)
  +0.07 Beavis and Butt-head Do America (1996)
  +0.06 Independence Day (ID4) (1996)
  +0.06 2001: A Space Odyssey (1968)

k = 50, moved up most:
  +0.13 Fantasia (1940)
  +0.11 People vs. Larry Flynt, The (1996)
  +0.11 Pulp Fiction (1994)
  +0.10 Mars Attacks! (1996)
  +0.10 Return of the Jedi (1983)
  +0.09 In & Out (1997)

Shared moved-up movies, k=2 and k=10: 2
Shared moved-up movies, k=2 and k=50: 1


**Answer:** The moved-up movies changed quite a bit when I changed k. At **k = 2**, *Pulp Fiction* and *Fargo* were near the top. At **k = 10**, *Fargo* and *Return of the Jedi* moved up most. At **k = 50**, the list started with *Fantasia* and *The People vs. Larry Flynt*.

Only **2 of the top 6** moved-up movies were shared between k = 2 and k = 10, and only **1 of the top 6** was shared between k = 2 and k = 50. A small k uses only a few general patterns, while a larger k learns more detailed ones. From Milestone 3, I also know that too many factors can start fitting noise, so using k = 50 is not automatically better.

## What we found

After I added my 10 ratings, Model 2 shared 9 of its top 10 picks with the baseline. SVD still raised my predicted rating for Fargo by 0.12 stars and lowered The English Patient by 0.11 stars.